# 01 - Pull the BRCA example out of the MATLAB workspace

Everything the example needs is inside `workspaceExample.mat`: the model, the FPKM matrix,
the gene ID dictionary, the medium settings, and the results MATLAB produced (useful as
ground truth). This notebook converts all of it into files under `data/`.


In [3]:
import json, pickle, pathlib
import numpy as np
import scipy.io as sio
import pandas as pd

import matlab_io as mio


BRCA_DIR = pathlib.Path(r"C:/Users/13371/Desktop/python project/rFastcormics_v2_python_1/BRCA_example")

WS = BRCA_DIR / "workspaceExample.mat"

OUT = pathlib.Path("data"); OUT.mkdir(exist_ok=True)
print(WS.resolve())

C:\Users\13371\Desktop\python project\rFastcormics_v2_python_1\BRCA_example\workspaceExample.mat


## Load the workspace

In [4]:
d = sio.loadmat(WS, struct_as_record=False)
sorted(k for k in d if not k.startswith("__"))

['BCRAmodel',
 'BCRAmodelBis',
 'BCRAmodelTer',
 'None',
 'ans',
 'biomassRxn',
 'colnames',
 'consensusProportion',
 'consistentModel',
 'consistentRxnsBool',
 'discretized',
 'epsilon',
 'fbaBCRA',
 'fbaControl',
 'fpkm',
 'geneListBCRA',
 'geneListControl',
 'grRateKOBCRA',
 'grRateKOControl',
 'grRateWTBCRA',
 'grRateWTControl',
 'grRatioBCRA',
 'grRatioControl',
 'mask',
 'model',
 'optionalSettings',
 'optionalSettingsBis',
 'retainedRxns',
 'retainedRxnsBis',
 'retainedRxnsTer',
 'rownames',
 'toRemove']

Inputs (`model`, `fpkm`, `rownames`, `colnames`, `dico`, `optionalSettings`) and results
(`BCRAmodel`, `retainedRxns`, `grRatioBCRA`, ...) are both in here.

## Model

In [5]:
model = mio.struct_to_model_dict(d["model"][0, 0])
print("S:", model["S"].shape)
print("rxns:", len(model["rxns"]), model["rxns"][:3])
print("mets:", len(model["mets"]), model["mets"][:3])
print("genes:", len(model["genes"]), model["genes"][:5])
print("rules:", [r for r in model["rules"][:40] if r][:3])
print("bounds:", model["lb"].min(), model["ub"].max())
print("objective:", model["rxns"][model["c"] != 0])

with open(OUT / "model.pkl", "wb") as fh:
    pickle.dump(model, fh, protocol=4)

S: (5837, 10601)
rxns: 10601 ['10FTHF5GLUtl' '10FTHF5GLUtm' '10FTHF6GLUtl']
mets: 5837 ['10fthf5glu[c]' '10fthf5glu[l]' '10fthf5glu[m]']
genes: 2248 ['8639.1' '26.1' '314.2' '314.1' '1591.1']
rules: ['x(1) | x(2) | x(3) | x(4)', 'x(5)', 'x(5)']
bounds: -1000.0 1000.0
objective: ['biomass_maintenance']


## Expression data

In [6]:
fpkm = np.asarray(d["fpkm"], float)
rownames = mio.cellstr(d["rownames"])          # gene IDs (SYMBOL)
colnames = mio.cellstr(d["colnames"])          # 25 TCGA BRCA samples
discretized_matlab = np.asarray(d["discretized"], float)

print(fpkm.shape, rownames[:4], colnames[:2], sep="\n")
print("MATLAB discretized:", dict(zip(*np.unique(discretized_matlab, return_counts=True))))

np.savez_compressed(OUT / "expression.npz", fpkm=fpkm, rownames=rownames,
                    colnames=colnames, discretized_matlab=discretized_matlab)

(23368, 25)
['1/2-SBSRNA4' 'A1BG' 'A1BG-AS1' 'A1CF']
['TCGA_A2_A3KC_01A_11R_A213_07' 'TCGA_A7_A6VW_01A_21R_A33J_07']
MATLAB discretized: {np.float64(-1.0): np.int64(183705), np.float64(0.0): np.int64(236880), np.float64(1.0): np.int64(163615)}


`discretized` is genes x samples with values 1 / 0 / -1 = expressed / unknown /
not expressed.

## dico (MATLAB table -> DataFrame)

In [7]:
dico = mio.read_matlab_table(WS)
dico.to_csv(OUT / "dico.csv", index=False)
print(dico.shape, list(dico.columns))
dico.head()

(42698, 2) ['ENTREZ', 'SYMBOL']


,ENTREZ,SYMBOL
0,1,A1BG
1,10,NAT2
2,100,ADA
3,1000,CDH2
4,10000,AKT3


Model genes are ENTREZ with a transcript suffix (`8639.1`), `rownames` are SYMBOLs.
`mapExpressionToModel` with `processTranscripts=1` strips the `.n` before matching.

## optionalSettings

In [8]:
def settings_of(struct):
    return {f: [str(x) for x in mio.cellstr(getattr(struct, f))] for f in struct._fieldnames}

settings = settings_of(d["optionalSettings"][0, 0])
settings_bis = settings_of(d["optionalSettingsBis"][0, 0])   # the insufficient-medium case

print({k: len(v) for k, v in settings.items()})
print("func:", settings["func"])
print("notMediumConstrained:", settings["notMediumConstrained"])
print("medium[:8]:", settings["medium"][:8])

with open(OUT / "optional_settings.json", "w") as fh:
    json.dump({"optionalSettings": settings, "optionalSettingsBis": settings_bis},
              fh, indent=1)

{'func': 3, 'notMediumConstrained': 1, 'medium': 54}
func: ['DM_atp_c_', 'biomass_reaction', 'biomass_maintenance']
notMediumConstrained: ['EX_hdl_hs[e]']
medium[:8]: ['gln_L[e]', 'gthrd[e]', 'o2[e]', 'glc_D[e]', 'pi[e]', 'cl[e]', 'na1[e]', 'k[e]']


## MATLAB results, for comparison later

In [9]:
ref = dict(
    consistentRxnsBool=np.asarray(d["consistentRxnsBool"], int).ravel(),   # 1-based
    retainedRxns=np.asarray(d["retainedRxns"], int).ravel(),
    retainedRxnsBis=np.asarray(d["retainedRxnsBis"], int).ravel(),
    retainedRxnsTer=np.asarray(d["retainedRxnsTer"], int).ravel(),
    BCRAmodel_rxns=mio.cellstr(d["BCRAmodel"][0, 0].rxns),
    BCRAmodel_genes=mio.cellstr(d["BCRAmodel"][0, 0].genes),
    BCRAmodelBis_rxns=mio.cellstr(d["BCRAmodelBis"][0, 0].rxns),
    BCRAmodelTer_rxns=mio.cellstr(d["BCRAmodelTer"][0, 0].rxns),
    biomassRxn=str(np.asarray(d["biomassRxn"]).ravel()[0]),
    consensusProportion=float(np.asarray(d["consensusProportion"]).ravel()[0]),
    epsilon=float(np.asarray(d["epsilon"]).ravel()[0]),
    # FBA / gene deletion results from the second half of tutorialScript.m
    fba_f_control=float(np.asarray(d["fbaControl"][0, 0].f).ravel()[0]),
    fba_f_BRCA=float(np.asarray(d["fbaBCRA"][0, 0].f).ravel()[0]),
    grRateWT_control=float(np.asarray(d["grRateWTControl"]).ravel()[0]),
    grRateWT_BRCA=float(np.asarray(d["grRateWTBCRA"]).ravel()[0]),
    grRatio_control=np.asarray(d["grRatioControl"], float).ravel(),
    grRateKO_control=np.asarray(d["grRateKOControl"], float).ravel(),
    grRatio_BRCA=np.asarray(d["grRatioBCRA"], float).ravel(),
    grRateKO_BRCA=np.asarray(d["grRateKOBCRA"], float).ravel(),
    # MATLAB's own context model bounds - lets notebook 04 check the downstream analysis
    # without the FASTCORE difference getting in the way
    BCRAmodel_lb=np.asarray(d["BCRAmodel"][0, 0].lb, float).ravel(),
    BCRAmodel_ub=np.asarray(d["BCRAmodel"][0, 0].ub, float).ravel(),
)
np.savez_compressed(OUT / "matlab_reference.npz", **ref)

print("fastcc kept:", ref["consistentRxnsBool"].size)
print("retainedRxns:", ref["retainedRxns"].size,
      "| Bis:", ref["retainedRxnsBis"].size, "| Ter:", ref["retainedRxnsTer"].size)
print("BCRAmodel:", ref["BCRAmodel_rxns"].size, "rxns /", ref["BCRAmodel_genes"].size, "genes")
print("growth rate  control:", ref["grRateWT_control"], " BRCA:", ref["grRateWT_BRCA"])
print("gene deletion  control:", ref["grRatio_control"].size,
      "genes  BRCA:", ref["grRatio_BRCA"].size, "genes")

fastcc kept: 10600
retainedRxns: 2584 | Bis: 2530 | Ter: 3126
BCRAmodel: 2584 rxns / 1290 genes
growth rate  control: 753.3364248443653  BRCA: 50.67169640681426
gene deletion  control: 1884 genes  BRCA: 1108 genes


In [10]:
sorted(p.name for p in OUT.iterdir())

['dico.csv',
 'discretized_python.npy',
 'expression.npz',
 'matlab_reference.npz',
 'model.pkl',
 'optional_settings.json']